# 03 — Modèles classiques (TF-IDF)

**Objectif de ce notebook** : entraîner et comparer **4 modèles classiques** de machine learning sur les textes nettoyés du notebook 02 (colonne `text_clean`) :
- Naive Bayes
- Régression logistique
- SVM linéaire
- Random Forest

Chaque modèle est évalué avec les mêmes critères : **accuracy**, **F1 macro**, **temps d'entraînement** et **taille du modèle**. Ces résultats seront comparés aux modèles de deep learning (notebook 04) dans le notebook 05, pour choisir le modèle à déployer.

## 0. Imports

- `time` : mesure des temps d'entraînement et de prédiction
- `io` + `joblib` : mesure de la taille d'un modèle une fois sauvegardé
- `TfidfVectorizer` : transformation des textes en vecteurs de nombres
- les 4 classifieurs de scikit-learn, `GridSearchCV` pour le réglage des hyperparamètres et les fonctions de métriques

In [ ]:
import io
import time
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

sns.set_theme(style="whitegrid")
pd.set_option("display.max_colwidth", 150)

## 1. Chargement des données nettoyées

On recharge les fichiers exportés à la fin du notebook 02. En local, ils sont lus dans le dossier `data/processed/` du dépôt ; sur Colab, ce dossier n'existe pas, donc ils sont téléchargés directement depuis GitHub. Le même notebook fonctionne ainsi dans les deux environnements, sans avoir à importer de fichier à la main.

In [ ]:
# En local : fichiers du dépôt. Sur Colab : téléchargement direct depuis GitHub.
DOSSIER_LOCAL = Path("../data/processed")
URL_GITHUB = "https://raw.githubusercontent.com/Akerman64/ag-news-classifier/main/data/processed"

source = DOSSIER_LOCAL if DOSSIER_LOCAL.exists() else URL_GITHUB
train_df = pd.read_csv(f"{source}/train_clean.csv.gz")
test_df = pd.read_csv(f"{source}/test_clean.csv.gz")

print("Source :", source)
print("Train :", train_df.shape, "| Test :", test_df.shape)

In [ ]:
train_df[["text_clean", "label_text"]].head()

In [ ]:
X_train_texte, y_train = train_df["text_clean"], train_df["label_text"]
X_test_texte, y_test = test_df["text_clean"], test_df["label_text"]
categories = sorted(y_train.unique())
categories

- **118 334 articles d'entraînement** et **7 425 de test**, répartis en 4 catégories.
- On utilise la colonne **`text_clean`** (minuscules, sans stopwords, lemmatisée) : c'est la version adaptée au TF-IDF, qui compte les mots.
- La cible est **`label_text`** (le nom de la catégorie) plutôt que le numéro : le modèle renverra directement « Sports », « World »… ce qui simplifie l'API.
- scikit-learn range les catégories par ordre alphabétique : Business, Sci/Tech, Sports, World.

## 2. Vectorisation TF-IDF

Un modèle ne sait pas lire du texte : il faut le transformer en nombres. Le **TF-IDF** attribue à chaque mot d'un article un poids :
- **TF** (*term frequency*) : le mot est-il fréquent **dans cet article** ?
- **IDF** (*inverse document frequency*) : le mot est-il rare **dans l'ensemble du corpus** ?

Un mot fréquent dans un article mais rare ailleurs (*touchdown*, *nasdaq*) reçoit un poids élevé : il est caractéristique. Un mot présent partout reçoit un poids faible.

Paramètres choisis :
- `ngram_range=(1, 2)` : on garde les mots seuls et les **paires de mots** (*wall street*, *prime minister*), qui ont un sens différent de leurs mots pris séparément
- `min_df=3` : on ignore les termes trop rares (fautes de frappe, noms propres uniques), qui n'aident pas à généraliser
- `max_features=100_000` : on limite la taille du vocabulaire, donc du modèle
- `sublinear_tf=True` : un mot répété 10 fois ne compte pas 10 fois plus qu'un mot présent une fois

⚠️ Le vectoriseur est entraîné (`fit`) **uniquement sur le train**, puis seulement appliqué (`transform`) au test : le test doit rester des textes « jamais vus », comme ceux que tapera un utilisateur de l'application.

In [ ]:
vectoriseur = TfidfVectorizer(
    ngram_range=(1, 2),     # mots seuls + paires de mots consécutifs
    min_df=3,               # ignore les termes présents dans moins de 3 articles
    max_features=100_000,   # garde les 100 000 termes les plus fréquents
    sublinear_tf=True,      # 1 + log(tf) : atténue les mots répétés
)

debut = time.time()
X_train = vectoriseur.fit_transform(X_train_texte)   # apprend le vocabulaire sur le train uniquement
X_test = vectoriseur.transform(X_test_texte)
temps_tfidf = time.time() - debut

print(f"Vectorisation : {temps_tfidf:.1f} s")
print("X_train :", X_train.shape, "| X_test :", X_test.shape)
print(f"Taux de remplissage : {X_train.nnz / (X_train.shape[0] * X_train.shape[1]) * 100:.3f} %")

In [ ]:
# Exemples de termes du vocabulaire : unigrammes et bigrammes
vocabulaire = vectoriseur.get_feature_names_out()
bigrammes = [t for t in vocabulaire if " " in t]
print("Unigrammes :", len(vocabulaire) - len(bigrammes), "| Bigrammes :", len(bigrammes))
print("Exemples de bigrammes :", bigrammes[::5000][:12])

- La vectorisation ne prend que **3,4 secondes** pour les 125 000 articles.
- On obtient une matrice de **118 334 lignes (articles) × 100 000 colonnes (termes)**.
- Elle n'est remplie qu'à **0,029 %** : chaque article ne contient en moyenne qu'une trentaine des 100 000 termes. scikit-learn utilise donc une **matrice creuse**, qui ne stocke que les valeurs non nulles. Sans cela, la matrice occuperait près de 95 Go en mémoire.
- Les **bigrammes représentent 77 % du vocabulaire** (77 157 contre 22 843 mots seuls). Certains apportent une vraie information (*police spokesman*, *cut rating*, *republican sen*), d'autres sont des associations fortuites (*charge took*). *aapl news* est un reste des liens boursiers de Reuters (code de l'action Apple).

## 3. Entraînement et évaluation

### 3.1 Fonction d'évaluation commune

Pour une comparaison équitable, tous les modèles passent par **la même fonction** : entraînement sur le train, prédiction sur le test, calcul des métriques.

Les critères mesurés :
- **Accuracy** : pourcentage d'articles bien classés
- **F1 macro** : moyenne des F1-scores des 4 catégories, chacune avec le même poids. Le F1 d'une catégorie combine la précision (les articles prédits « Sports » sont-ils bien du sport ?) et le rappel (retrouve-t-on tous les articles de sport ?). C'est notre **métrique principale** : elle pénalise un modèle qui négligerait une catégorie.
- **Temps d'entraînement** et **temps de prédiction par article** (en microsecondes)
- **Taille** : vectoriseur TF-IDF + classifieur sauvegardés avec `joblib`. On compte les deux, car l'API aura besoin des deux pour classer un texte : c'est la taille réelle du modèle déployé.

In [ ]:
def taille_mo(objet):
    """Taille de l'objet une fois sauvegardé avec joblib, en Mo."""
    tampon = io.BytesIO()
    joblib.dump(objet, tampon)
    return tampon.getbuffer().nbytes / 1e6


resultats = []
predictions = {}

def evaluer(nom, modele):
    debut = time.time()
    modele.fit(X_train, y_train)
    temps_entrainement = time.time() - debut

    debut = time.time()
    y_pred = modele.predict(X_test)
    temps_prediction_us = (time.time() - debut) / len(y_test) * 1e6   # temps moyen par article, en microsecondes

    predictions[nom] = y_pred
    resultats.append({
        "modele": nom,
        "accuracy": accuracy_score(y_test, y_pred),
        "f1_macro": f1_score(y_test, y_pred, average="macro"),
        "temps_entrainement_s": temps_entrainement,
        "prediction_us_par_article": temps_prediction_us,
        "taille_mo": taille_mo(vectoriseur) + taille_mo(modele),   # le modèle déployé = TF-IDF + classifieur
    })
    r = resultats[-1]
    print(f"{nom} : accuracy = {r['accuracy']:.4f}, F1 macro = {r['f1_macro']:.4f}, "
          f"entraînement = {temps_entrainement:.1f} s, taille = {r['taille_mo']:.1f} Mo")

La fonction ne renvoie rien : elle ajoute une ligne au tableau `resultats` et garde les prédictions dans `predictions`, réutilisées plus bas pour les matrices de confusion et l'analyse des erreurs.

### 3.2 Réglage des hyperparamètres (validation croisée)

Chaque modèle a des **hyperparamètres** : des réglages choisis avant l'entraînement, qui ne sont pas appris à partir des données.
- `alpha` (Naive Bayes) : lissage, qui évite une probabilité nulle pour un mot jamais vu dans une catégorie
- `C` (régression logistique et SVM) : inverse de la régularisation. Un `C` petit force un modèle plus simple, qui colle moins au train et généralise mieux ; un `C` grand laisse le modèle s'ajuster davantage, au risque de surapprendre.

`GridSearchCV` teste chaque valeur par **validation croisée à 3 plis** : le train est découpé en 3 parts, le modèle est entraîné sur 2 et évalué sur la 3e, trois fois. Le choix se fait **sans jamais toucher au test**, sinon le score final serait trop optimiste.

Le Random Forest n'est pas réglé : son entraînement est beaucoup plus long, on garde ses paramètres par défaut (100 arbres).

In [ ]:
grilles = {
    "Naive Bayes": (MultinomialNB(), {"alpha": [0.05, 0.1, 0.5, 1.0]}),
    "Régression logistique": (LogisticRegression(max_iter=1000), {"C": [1, 5, 10]}),
    "SVM linéaire": (LinearSVC(), {"C": [0.1, 0.5, 1]}),
}

meilleurs_params = {}
for nom, (modele, grille) in grilles.items():
    debut = time.time()
    recherche = GridSearchCV(modele, grille, cv=3, scoring="f1_macro", n_jobs=-1)
    recherche.fit(X_train, y_train)
    meilleurs_params[nom] = recherche.best_params_
    scores = dict(zip([str(p) for p in recherche.cv_results_["params"]],
                      recherche.cv_results_["mean_test_score"].round(4)))
    print(f"{nom} ({time.time() - debut:.0f} s) : {scores} → meilleur : {recherche.best_params_}")

- **Naive Bayes** : `alpha = 0.1`, mais les scores sont presque identiques quelle que soit la valeur (0.8957 à 0.8963) : ce modèle est peu sensible à ce réglage.
- **Régression logistique** : `C = 1` (0.8973). Le score baisse quand `C` augmente (0.8907 pour `C = 10`) : le modèle commence à surapprendre.
- **SVM linéaire** : `C = 0.1` (0.9002), la valeur la plus forte en régularisation testée.
- Pour les deux modèles linéaires, **la régularisation la plus forte gagne** : avec 100 000 termes pour 118 000 articles, il est facile de « par cœur » apprendre le train, et il faut contraindre les modèles pour qu'ils généralisent.
- Toute la recherche prend moins de 20 secondes : ces modèles sont très rapides.

### 3.3 Naive Bayes

**Naive Bayes multinomial** : modèle probabiliste fondé sur le théorème de Bayes. Pour chaque catégorie, il apprend la probabilité d'apparition de chaque mot, puis choisit la catégorie la plus probable pour un nouveau texte. Il suppose (naïvement) que les mots sont indépendants les uns des autres. C'est la **référence historique** en classification de textes : simple, très rapide, souvent étonnamment bon.

In [ ]:
evaluer("Naive Bayes", MultinomialNB(**meilleurs_params["Naive Bayes"]))

### 3.4 Régression logistique

**Régression logistique** : modèle linéaire qui apprend **un poids par terme et par catégorie**. Pour classer un texte, il additionne les poids de ses termes et transforme les scores en **probabilités** (fonction softmax). Avantage pour l'application : il donne nativement la probabilité de chaque catégorie (`predict_proba`).

In [ ]:
evaluer("Régression logistique", LogisticRegression(max_iter=1000, **meilleurs_params["Régression logistique"]))

### 3.5 SVM linéaire

**SVM linéaire** (*Support Vector Machine*) : cherche les frontières qui séparent le mieux les catégories, en maximisant la **marge** entre elles. Très efficace sur les données en grande dimension et creuses comme le TF-IDF. Limite : `LinearSVC` ne donne **pas de probabilités**, seulement la catégorie prédite.

In [ ]:
evaluer("SVM linéaire", LinearSVC(**meilleurs_params["SVM linéaire"]))

### 3.6 Random Forest

**Random Forest** : ensemble de 100 **arbres de décision**, chacun entraîné sur un échantillon aléatoire des articles et des termes ; la prédiction finale est le vote majoritaire des arbres. Très performant sur des données tabulaires, il est moins adapté au texte : chaque nœud d'un arbre ne regarde qu'un seul terme parmi 100 000, et la plupart des termes valent 0 dans un article donné.

In [ ]:
evaluer("Random Forest", RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=42))

- **SVM linéaire** : le meilleur, **F1 macro = 0.911**, entraîné en moins de 2 secondes.
- **Régression logistique** : juste derrière (0.909).
- **Naive Bayes** : 0.902, entraîné en un dixième de seconde.
- **Random Forest** : le moins bon (0.890), **le plus lent** (environ 24 s) et surtout **énorme : 612 Mo**.
- Les scores sur le test sont proches, et même un peu supérieurs, à ceux de la validation croisée (0.900 pour le SVM) : les modèles finaux sont entraînés sur tout le train (et non sur 2/3), et **il n'y a pas de surapprentissage**.

*Les temps varient légèrement d'une exécution à l'autre et dépendent de la machine (ils seront plus longs sur Colab, qui n'a que 2 cœurs).*

## 4. Comparaison des modèles

### 4.1 Tableau récapitulatif

Tous les résultats rassemblés dans un tableau, triés par F1 macro.

In [ ]:
comparaison = pd.DataFrame(resultats).set_index("modele").sort_values("f1_macro", ascending=False)
comparaison.round(4)

- Les **3 modèles linéaires sont très proches** : 0.9 point d'écart entre le SVM et Naive Bayes, et 0.2 point entre le SVM et la régression logistique, soit une quinzaine d'articles sur 7 425.
- **Prédiction** : environ **0,15 microseconde par article** pour les modèles linéaires, donc négligeable. Le Random Forest est **près de 90 fois plus lent** (12 µs).
- **Taille** : 7,4 Mo pour le SVM et la régression logistique, dont environ 4,2 Mo pour le vocabulaire TF-IDF et 3,2 Mo pour les poids (4 catégories × 100 000 termes). Naive Bayes pèse 10,6 Mo car il stocke deux tables (comptages et probabilités).

### 4.2 Performances

Comparaison visuelle de l'accuracy et du F1 macro.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5))
comparaison[["accuracy", "f1_macro"]].plot.bar(ax=ax, color=["steelblue", "darkorange"], rot=0)
ax.set_ylim(0.8, 0.95)
ax.set_title("Accuracy et F1 macro sur le jeu de test")
ax.set_xlabel("")
ax.set_ylabel("Score")
for conteneur in ax.containers:
    ax.bar_label(conteneur, fmt="%.3f", fontsize=8)
plt.tight_layout()
plt.show()

- ⚠️ L'axe vertical commence à 0.80 pour rendre les écarts visibles : visuellement ils paraissent grands, mais en réalité **l'écart maximal est de 2 points** (SVM contre Random Forest).
- Pour chaque modèle, **accuracy et F1 macro sont quasi égaux** : c'est normal avec des catégories équilibrées (environ 25 % chacune). Avec des catégories déséquilibrées, le F1 macro serait plus bas et révélerait les catégories mal reconnues.

### 4.3 Coût : temps d'entraînement et taille

La performance n'est pas le seul critère : l'application sera déployée gratuitement sur Vercel, avec une **limite de 250 Mo** pour l'ensemble du code et des dépendances. Le temps d'entraînement et la taille du modèle comptent donc aussi.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

comparaison["temps_entrainement_s"].plot.bar(ax=axes[0], color="seagreen", rot=0, logy=True)
axes[0].set_title("Temps d'entraînement (échelle logarithmique)")
axes[0].set_ylabel("Secondes")
axes[0].bar_label(axes[0].containers[0], fmt="%.1f s")

comparaison["taille_mo"].plot.bar(ax=axes[1], color="indianred", rot=0, logy=True)
axes[1].set_title("Taille du modèle déployé : TF-IDF + classifieur (échelle log.)")
axes[1].set_ylabel("Mo")
axes[1].bar_label(axes[1].containers[0], fmt="%.1f Mo")

for ax in axes:
    ax.set_xlabel("")
plt.tight_layout()
plt.show()

- Une **échelle logarithmique** est nécessaire tant les écarts sont grands : le Random Forest met environ **200 fois plus de temps** à s'entraîner que Naive Bayes.
- **Taille : 612 Mo pour le Random Forest**, plus de 80 fois le SVM. Ses 100 arbres sont entièrement développés et chacun stocke des milliers de nœuds. C'est **plus du double de la limite de Vercel** : le Random Forest est éliminé pour le déploiement, et il est en plus le moins performant.
- Les modèles linéaires, eux, sont à la fois **les plus performants et les plus légers**.

### 4.4 Matrices de confusion

La matrice de confusion montre, pour chaque vraie catégorie (ligne), comment les articles ont été classés (colonnes). Elle est normalisée par ligne : chaque case donne le **pourcentage des articles de la vraie catégorie**, et la diagonale correspond aux bonnes prédictions.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
for ax, nom in zip(axes.ravel(), comparaison.index):
    matrice = confusion_matrix(y_test, predictions[nom], labels=categories, normalize="true") * 100
    sns.heatmap(matrice, annot=True, fmt=".1f", cmap="Blues", cbar=False, ax=ax,
                xticklabels=categories, yticklabels=categories)
    ax.set_title(f"{nom} (% de chaque vraie catégorie)")
    ax.set_xlabel("Catégorie prédite")
    ax.set_ylabel("Vraie catégorie")
plt.tight_layout()
plt.show()

Les 4 modèles font **les mêmes types d'erreurs** :
- **Sports est la catégorie la mieux reconnue** (97 à 98,5 %) : son vocabulaire est très spécifique (*game*, *season*, *coach*, noms d'équipes).
- **La principale confusion est Business ↔ Sci/Tech** : pour le SVM, 8,5 % des articles Business sont prédits Sci/Tech, et 7,3 % des Sci/Tech prédits Business. Les articles sur les entreprises technologiques (Apple, Google, IBM, Microsoft) relèvent des deux catégories.
- **World est surtout confondu avec Business** (4,9 %) : politique économique, pétrole, commerce international.
- Naive Bayes est le plus touché par la confusion Business → Sci/Tech (10,8 %).

### 4.5 Rapport détaillé du meilleur modèle

Le rapport de classification détaille les métriques **par catégorie** pour le meilleur modèle :
- **precision** : parmi les articles prédits dans cette catégorie, le pourcentage qui y appartient vraiment
- **recall** (rappel) : parmi les articles de cette catégorie, le pourcentage retrouvé par le modèle
- **f1-score** : la combinaison des deux
- **support** : le nombre d'articles de cette catégorie dans le test

In [ ]:
meilleur = comparaison.index[0]
print("Meilleur modèle :", meilleur)
print(classification_report(y_test, predictions[meilleur], digits=4))

Pour le SVM linéaire :
- **Sports** : F1 = 0.968, avec un rappel de 98,5 % : presque tous les articles de sport sont retrouvés.
- **Business (0.877) et Sci/Tech (0.887)** sont les catégories les plus difficiles, ce qui confirme la confusion vue dans la matrice.
- **World** : précision de 93,4 % mais rappel de 89,5 %. Quand le modèle prédit World, il a presque toujours raison, mais il « rate » environ 1 article World sur 10, classé en Business ou en Sports.

### 4.6 Exemples d'erreurs

Regardons quelques articles mal classés par le SVM, pour comprendre la nature des erreurs.

In [ ]:
masque_erreurs = predictions[meilleur] != y_test
erreurs = test_df.loc[masque_erreurs, ["text", "label_text"]].copy()
erreurs["text"] = erreurs["text"].str[:110]   # texte raccourci pour la lisibilité
erreurs["prediction"] = predictions[meilleur][masque_erreurs]
print(f"{len(erreurs)} erreurs sur {len(test_df)} articles")
erreurs.sample(8, random_state=0)

- **654 erreurs sur 7 425 articles** (8,8 %).
- Plusieurs « erreurs » sont en fait des **erreurs d'étiquetage du dataset**, où le modèle a raison :
  - *Colts Lead Pats Early in Third Quarter* (match de football américain) est étiqueté **World**, et le modèle prédit **Sports** ;
  - *Cricket: Dubai global academy* est étiqueté World, prédit Sports ;
  - *Lazarus-like virus hits computers* (un virus informatique) est étiqueté Business, prédit Sci/Tech.
- D'autres articles sont **réellement ambigus** : *Indian PM pledges to protect poor from oil-driven inflation* parle d'un Premier ministre (World) et d'inflation (Business).
- **Conclusion** : une partie des erreurs est irréductible, car liée au bruit dans les étiquettes d'AG News. Aucun modèle ne pourra atteindre 100 %.

## 5. Export des métriques

Les métriques sont sauvegardées dans `models/metrics_classiques.csv`, avec les mêmes colonnes que `metrics_deep.csv` (notebook 04). Le notebook 05 rassemblera les deux fichiers pour comparer les 6 modèles.

Le modèle lui-même n'est **pas encore exporté** : le choix final se fera dans le notebook 05, après comparaison avec le deep learning.

In [ ]:
DOSSIER_MODELES = Path("../models") if Path("../models").exists() else Path("models")
DOSSIER_MODELES.mkdir(exist_ok=True)

colonnes = ["accuracy", "f1_macro", "temps_entrainement_s", "taille_mo"]
comparaison[colonnes].round(4).to_csv(DOSSIER_MODELES / "metrics_classiques.csv")
print(open(DOSSIER_MODELES / "metrics_classiques.csv").read())

## 6. Bilan

| Modèle | F1 macro | Entraînement | Taille | Bilan |
|---|---|---|---|---|
| **SVM linéaire** | **0.911** | ~2 s | 7,4 Mo | Le meilleur, mais pas de probabilités |
| Régression logistique | 0.909 | ~3 s | 7,4 Mo | Quasi aussi bon, probabilités natives |
| Naive Bayes | 0.902 | 0,1 s | 10,6 Mo | Très rapide, un peu moins précis |
| Random Forest | 0.890 | ~24 s | 612 Mo | Le moins bon et trop lourd pour Vercel ❌ |

- Les **modèles linéaires sur TF-IDF** sont les mieux adaptés à ce problème : performants, rapides et légers.
- **Candidats au déploiement** : le SVM linéaire (en l'enveloppant dans `CalibratedClassifierCV` pour obtenir des probabilités) ou la régression logistique.
- La principale difficulté est la frontière **Business / Sci/Tech**, et une partie des erreurs vient d'étiquettes incorrectes dans le dataset.

**Suite** : les modèles de deep learning (LSTM et DistilBERT, notebook 04) feront-ils mieux, et à quel coût ?